In [ ]:
# Getting imports and the model setup
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from smolagents import OpenAIModel
from dotenv import load_dotenv
load_dotenv()

model_name = "gpt-5.4-mini"
model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
print(f"Using model: {model_name}")

In [ ]:
# Getting the tools setup and the MarkovReActCodeAgent setup
import sys
sys.path.insert(0, "../baseline")

from smolagents.monitoring import LogLevel
from common_setup import build_tools
from smolagents.markov_react import MarkovReActCodeAgent

tools, ti_tool, visualizer = build_tools(model)

window_size = 3
agent = MarkovReActCodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    window_size=window_size,
)

In [ ]:
# Load GAIA validation set from HuggingFace
# Visit https://huggingface.co/datasets/gaia-benchmark/GAIA to request access first
import pandas as pd
from common_setup import evaluate_agent, load_gaia_dataset, question_scorer

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")
print(pd.DataFrame(eval_ds)["task"].value_counts())

In [ ]:
results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=20,
    output_file=f"markov_react_{model_name}_w{window_size}.jsonl",
    pickle_dir=f"markov_react_{model_name}_w{window_size}",
)

In [ ]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print(f"=== MarkovReAct GAIA Evaluation Results (window_size={window_size}) ===")
print(f"Overall accuracy:   {correct}/{total} = {correct/total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg tokens per question:  {total_tokens:,.0f}")

print(f"\nAccuracy by level:")
for level in sorted(df["task"].unique()):
    level_df = df[df["task"] == level]
    lc = level_df["is_correct"].sum()
    lt = len(level_df)
    print(f"  Level {level}: {lc}/{lt} = {lc/lt:.1%}")